# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset object
dataset = mlc.Dataset(croissant_url)

# Display basic metadata
print(f"Dataset name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"License: {dataset.metadata.license}\n")
print(f"Citation: {dataset.metadata.cite_as}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @id
print("Record Sets available in the dataset:")
for record_set in dataset.metadata.record_sets:
    print(f"  - name: {record_set.name} | @id: {record_set.id}")

# We'll focus on the primary record set.
record_sets = dataset.metadata.record_sets
main_record_set = record_sets[0]
record_set_id = main_record_set.id

print(f"\nFields for Record Set '{main_record_set.name}' (@id: {main_record_set.id}):")
for field in main_record_set.fields:
    print(f"  - {field.name} (@id: {field.id}) | Data Type: {field.data_type}")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Load all records from the identified record set using its @id
df = pd.DataFrame(dataset.records(record_set=record_set_id))
print(f"Loaded {len(df)} records from {main_record_set.name} (@id: {record_set_id})\n")
print("Fields (columns) available:")
print(list(df.columns))

df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, and grouping data by key attributes.

In [ ]:
# Identify numeric fields (from the field overview above). For demonstration, we'll use 'cr:mw' as the molecular weight column (@id should match field list!).
# You can change 'cr:mw' to correspond to any available numeric field @id.
numeric_field_id = None
for field in main_record_set.fields:
    # Try to find a relevant numeric field (float or int)
    if field.data_type.lower() in ['float', 'integer', 'number', 'double']:
        numeric_field_id = field.id
        print(f"Using numeric field: {field.name} (@id: {field.id}) of type {field.data_type}")
        break
if numeric_field_id is None:
    raise RuntimeError("No numeric field found in record set fields.")

# Filter: Remove records with missing/invalid data and filter for values greater than a threshold
threshold = df[numeric_field_id].quantile(0.25)  # Example: Keep values above 25th percentile
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold}\nResulting DataFrame shape: {filtered_df.shape}\n")

# Normalize the chosen numeric field (Z-score normalization)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head()

In [ ]:
# Group by a relevant categorical field, e.g., 'cr:description' if available (or pick another string field from the overview).
group_field_id = None
for field in main_record_set.fields:
    # Try to find a categorical field
    if field.data_type.lower() in ['string', 'text'] and field.id != numeric_field_id:
        group_field_id = field.id
        print(f"Using group field: {field.name} (@id: {field.id}) of type {field.data_type}")
        break

if group_field_id is not None and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped by {group_field_id}. Mean {numeric_field_id} per category:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field (e.g. molecular weight)
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field_id], kde=True, bins=30)
plt.title(f'Distribution of {numeric_field_id} (filtered)')
plt.xlabel(numeric_field_id)
plt.show()

# If grouping was possible, show group means
if group_field_id is not None and group_field_id in filtered_df.columns:
    plt.figure(figsize=(10, 4))
    # Show top 20 (if many groups)
    top_groups = grouped_df.sort_values(numeric_field_id, ascending=False).head(20)
    sns.barplot(data=top_groups, x=group_field_id, y=numeric_field_id, palette='Blues_d')
    plt.title(f'Mean {numeric_field_id} per {group_field_id} (top 20)')
    plt.xticks(rotation=90)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the dataset via its Croissant schema using `mlcroissant` and inspected metadata, fields, and records using their respective `@id`s.
- We demonstrated field-based filtering and normalization for numeric fields, and grouped values by categorical attributes where possible.
- Data visualizations provide insights into the distribution and grouped aggregates for selected fields.

**For further work, check the dataset's Croissant schema for more specialized field analyses, data merging, or advanced modeling.**